# 5.3 FastAPI — Building Data APIs> ☕ **Java parallel:** Like Spring Boot, but 3x less boilerplate. Auto-generates OpenAPI docs.---

In [ ]:
# Save as app.py and run with: uvicorn app:app --reloadfastapi_code = '''from fastapi import FastAPI, HTTPException, Queryfrom pydantic import BaseModelfrom datetime import datetimeapp = FastAPI(    title="Healthcare Data API",    description="API for patient data and analytics",    version="1.0.0",)# Pydantic models for request/response — like Java DTOsclass Patient(BaseModel):    patient_id: str    name: str    age: int    diagnosis: str | None = Noneclass PipelineStatus(BaseModel):    pipeline_name: str    status: str    records_processed: int    last_run: datetime# In-memory store (use a DB in production)patients_db: dict[str, Patient] = {}@app.get("/")def health_check():    return {"status": "healthy", "timestamp": datetime.now()}@app.post("/patients/", response_model=Patient, status_code=201)def create_patient(patient: Patient):    """Create a new patient record."""    if patient.patient_id in patients_db:        raise HTTPException(status_code=409, detail="Patient already exists")    patients_db[patient.patient_id] = patient    return patient@app.get("/patients/{patient_id}", response_model=Patient)def get_patient(patient_id: str):    """Get patient by ID."""    if patient_id not in patients_db:        raise HTTPException(status_code=404, detail="Patient not found")    return patients_db[patient_id]@app.get("/patients/", response_model=list[Patient])def list_patients(    diagnosis: str | None = Query(None, description="Filter by diagnosis"),    min_age: int = Query(0, ge=0),):    """List patients with optional filters."""    results = list(patients_db.values())    if diagnosis:        results = [p for p in results if p.diagnosis == diagnosis]    results = [p for p in results if p.age >= min_age]    return results'''print(fastapi_code)print("\n# Run: uvicorn app:app --reload")print("# Docs: http://localhost:8000/docs (auto-generated Swagger UI)")